# Fine-tune Laya on kommentarwache's exported decisions

Runs on Kaggle (2x T4) or any machine with a GPU. Input: `train.jsonl` from `kw dataset export`.
Output: a checkpoint directory with fitted temperatures, to copy to `training/checkpoints/<name>` on the server.

The export's lines are `{state, questions, expected, meta}` where `meta.split` is `train`, `calib` or `test`.
`expected` holds, per question, the chosen option and a probability distribution (hard for human labels, the teacher's for the rest).

In [ ]:
%pip install -q laya
import json, pathlib
EXPORT = pathlib.Path('/kaggle/input/kommentarwache/train.jsonl')  # adjust
OUT = pathlib.Path('/kaggle/working/laya-kommentarwache')
rows = [json.loads(l) for l in EXPORT.read_text(encoding='utf8').splitlines() if l.strip()]
by_split = {s: [r for r in rows if r['meta']['split'] == s] for s in ('train', 'calib', 'test')}
{k: len(v) for k, v in by_split.items()}

## Shape check

Laya's trainer reads `{state, questions, expected}`. Confirm the installed version's expected
shape for soft targets here (`laya-train --help`, or `laya.training` docs) and adapt `to_record` if
it wants hard labels only: then use `expected[q]['choice']`.

In [ ]:
def to_record(r, soft=True):
    expected = {q: (a if soft else a['choice']) for q, a in r['expected'].items()}
    return {'state': r['state'], 'questions': r['questions'], 'expected': expected}

for split, items in by_split.items():
    with open(f'/kaggle/working/{split}.jsonl', 'w', encoding='utf8') as f:
        for r in items:
            f.write(json.dumps(to_record(r), ensure_ascii=False) + '\n')
!head -c 600 /kaggle/working/train.jsonl

In [ ]:
# Fine-tune the multilingual checkpoint (German and English comments).
!laya-train --data /kaggle/working/train.jsonl --base convaiinnovations/laya-multilingual --out {OUT} --epochs 3

In [ ]:
# Calibrate on the calib split: temperatures per question type and option count, saved with the checkpoint.
import laya
agent = laya.load(str(OUT))
calib = [json.loads(l) for l in open('/kaggle/working/calib.jsonl', encoding='utf8')]
records = laya.records_from_labeled(agent, calib)  # see the installed version's docs for the exact helper
laya.fit_temperatures(agent, records)
laya.save_calibration(agent, str(OUT))

In [ ]:
# Quick accuracy on the frozen test split before the server-side gate.
test = [json.loads(l) for l in open('/kaggle/working/test.jsonl', encoding='utf8')]
hits = total = 0
for r in test:
    answers = agent.predict(r['state'], r['questions'])['answers']
    for q, exp in r['expected'].items():
        want = exp['choice'] if isinstance(exp, dict) else exp
        total += 1
        hits += answers[q].get('choice') == want
print(f'per-question accuracy on test: {hits}/{total} = {hits/total:.3f}')

Download the `OUT` directory, place it at `training/checkpoints/<name>` on the server, start `laya-serve`
with `LAYA_MODELS=<name>=/checkpoints/<name>`, then `kw laya gate --source <gold source> --model <name>`.